### **[Unique Path (LeetCode 62)](https://leetcode.com/problems/unique-paths/description/)**

Imagine you are standing at the top-left corner of a city grid, and your home is at the bottom-right corner. You can only walk forward (Right) or cross the street downward (Down). You are not allowed to walk diagonally, backwards, or upwards. The question asks: how many completely unique routes can you take to get from your starting point to your home?

In simple terms: We need to count every possible valid path from `(0, 0)` to `(m-1, n-1)` moving only Right and Down.

---

### **Constraints Analysis**

* `1 <= m, n <= 100`: The grid can be up to 100x100.
* **Time Limitation:** While the grid itself isn't massive, the *number of possible paths* grows exponentially. A naive recursive approach (making a choice to go Right or Down at every single step) results in an $O(2^{m+n})$ time complexity. For a 100x100 grid, this would take longer than the universe has existed. We need an $O(m \times n)$ solution.
* **Space Limitation:** We want to avoid blowing up the call stack with deep recursion. An iterative approach using $O(m \times n)$ auxiliary space is expected, but an optimal approach will reduce this even further.

---

### **Approach Selection**

* **Pattern:** Dynamic Programming (DP) / Tabulation.
* **Data Structure:** 2D Array (and eventually a 1D Array).
* **Reasoning:** Let's look at a specific intersection on the grid, say cell `(r, c)`. Because the robot can *only* move Right or Down, there are only two possible ways the robot could have ever arrived at `(r, c)`:
1. It stepped Down from the cell directly above it: `(r - 1, c)`.
2. It stepped Right from the cell directly to its left: `(r, c - 1)`.


Therefore, the total number of unique paths to reach `(r, c)` is simply the sum of the paths to reach the cell above it PLUS the paths to reach the cell to its left. This is the hallmark of overlapping subproblems, perfect for Dynamic Programming.

---

### **Step-by-Step Implementation**

#### **1. Brute Force Approach (Pure Recursion)**

We write a function that explores every single valid move until it hits the destination.

```python
class Solution:
    def uniquePaths(self, m: int, n: int) -> int:
        def dfs(r, c):
            # Base Case 1: Reached the destination, this is 1 valid path!
            if r == m - 1 and c == n - 1:
                return 1
            # Base Case 2: Out of bounds, invalid path
            if r >= m or c >= n:
                return 0
                
            # The total paths from here is the sum of going Down and going Right
            return dfs(r + 1, c) + dfs(r, c + 1)
            
        return dfs(0, 0)

```

* **Time Complexity:** $O(2^{m+n})$. We are branching into two paths at nearly every step, calculating the same intersections over and over again. This will result in a Time Limit Exceeded (TLE).
* **Space Complexity:** $O(m + n)$. The maximum depth of the recursive call stack.
* **Interview Reasoning:** Mention this briefly to prove you understand the recursive relationship and base cases, but immediately state that it calculates overlapping subproblems redundantly, prompting the need for memoization or tabulation.

---

#### **2. Better Approach (2D Dynamic Programming)**

Instead of recalculating, we build a 2D grid and fill it out iteratively. The top row and left column only have 1 possible path (you can only get to them by walking straight Right or straight Down from the start). We fill the rest of the grid using our rule: `paths[r][c] = paths[r-1][c] + paths[r][c-1]`.

```python
class Solution:
    def uniquePaths(self, m: int, n: int) -> int:
        # Create an m x n grid filled with 1s
        dp = [[1] * n for _ in range(m)]
        
        # Start iterating from row 1 and col 1 
        # (since row 0 and col 0 are already correct with 1s)
        for r in range(1, m):
            for c in range(1, n):
                # The current cell is the sum of the cell above and the cell to the left
                dp[r][c] = dp[r - 1][c] + dp[r][c - 1]
                
        # The bottom-right corner contains our final answer
        return dp[m - 1][n - 1]

```

* **Time Complexity:** $O(m \times n)$. We visit every cell in the grid exactly once.
* **Space Complexity:** $O(m \times n)$. We allocate a full 2D array to store the paths.
* **Interview Reasoning:** This is a fantastic, fully functional solution. However, an interviewer will look at the line `dp[r][c] = dp[r - 1][c] + dp[r][c - 1]` and ask: *"Notice how you only ever look at the current row and the row immediately above it? Do we really need to store the entire grid in memory?"*

---

#### **3. Optimal Approach (1D Space-Optimized DP)**

We can optimize the space by collapsing our 2D grid into a single 1D array. As we iterate through the columns, the value currently sitting in our 1D array represents the cell "above" us, and the value we just updated to the left of our current pointer represents the cell to our "left".

```python
class Solution:
    def uniquePaths(self, m: int, n: int) -> int:
        # We only need to keep track of a single row at a time.
        # Initialize the first row with 1s.
        row = [1] * n
        
        # Process the remaining m - 1 rows
        for i in range(1, m):
            # The first element of any new row is always 1 (can only go strictly down)
            # So we start updating from column 1
            for j in range(1, n):
                # row[j] is the cell ABOVE us.
                # row[j-1] is the new updated cell to the LEFT of us.
                row[j] = row[j] + row[j - 1]
                
        # The last element in our row array is the final destination
        return row[-1]

```

* **Time Complexity:** $O(m \times n)$. We still process the equivalent of all cells.
* **Space Complexity:** $O(n)$. We strictly allocate a single array of size $n$, massively reducing our memory footprint.
* **Interview Reasoning:** This is the optimal DP solution. It shows a deep understanding of state reduction. *Bonus Flex:* If you want to really impress an interviewer, mention that this problem can be solved in $O(m+n)$ time and $O(1)$ space using pure Combinatorics (calculating "m+n-2 choose n-1"), but emphasize that the 1D DP approach is the expected algorithmic answer because it's extensible if the interviewer decides to add obstacles to the grid later.